# Monitoring — retraining trigger logic

Decision function a scheduler (cron / Airflow sensor) would call daily, passing in current state pulled from the monitoring store. Not wired to a real scheduler in this project — evaluated here against a few example scenarios to demonstrate the logic.

Three independent signals, any one of which is sufficient to trigger a retrain: (1) enough new labeled data has accumulated (routine refresh), (2) AUC on recent labeled feedback has dropped meaningfully vs. the model's validation AUC at promotion time, (3) the drift check has fired repeatedly.

In [1]:
from dataclasses import dataclass

@dataclass
class RetrainSignals:
    days_since_last_train: int
    new_labeled_rows_since_last_train: int
    current_model_auc_at_promotion: float
    recent_feedback_auc: float | None  # None if not enough labeled feedback yet
    consecutive_drift_flags: int

# Thresholds - tune based on observed business cadence, not fixed forever
MIN_NEW_ROWS_FOR_RETRAIN = 500
MAX_AUC_DROP_ALLOWED = 0.03
MAX_CONSECUTIVE_DRIFT_FLAGS = 3


## Decision function

In [2]:
def should_retrain(signals):
    reasons = []
    if signals.new_labeled_rows_since_last_train >= MIN_NEW_ROWS_FOR_RETRAIN:
        reasons.append(
            f'volume: {signals.new_labeled_rows_since_last_train} new rows '
            f'>= threshold {MIN_NEW_ROWS_FOR_RETRAIN}')
    if signals.recent_feedback_auc is not None:
        auc_drop = signals.current_model_auc_at_promotion - signals.recent_feedback_auc
        if auc_drop >= MAX_AUC_DROP_ALLOWED:
            reasons.append(
                f'performance: AUC dropped {auc_drop:.3f} '
                f'(promotion={signals.current_model_auc_at_promotion}, '
                f'recent={signals.recent_feedback_auc}) >= threshold {MAX_AUC_DROP_ALLOWED}')
    if signals.consecutive_drift_flags >= MAX_CONSECUTIVE_DRIFT_FLAGS:
        reasons.append(
            f'drift: {signals.consecutive_drift_flags} consecutive drift flags '
            f'>= threshold {MAX_CONSECUTIVE_DRIFT_FLAGS}')
    return {'retrain': len(reasons) > 0, 'reasons': reasons}


## Example scenarios

In [3]:
examples = [
    RetrainSignals(days_since_last_train=10, new_labeled_rows_since_last_train=120,
                    current_model_auc_at_promotion=0.853, recent_feedback_auc=0.849,
                    consecutive_drift_flags=0),
    RetrainSignals(days_since_last_train=30, new_labeled_rows_since_last_train=600,
                    current_model_auc_at_promotion=0.853, recent_feedback_auc=0.847,
                    consecutive_drift_flags=1),
    RetrainSignals(days_since_last_train=20, new_labeled_rows_since_last_train=200,
                    current_model_auc_at_promotion=0.853, recent_feedback_auc=0.79,
                    consecutive_drift_flags=4),
]
for i, sig in enumerate(examples, 1):
    result = should_retrain(sig)
    print(f"Scenario {i}: retrain={result['retrain']}  reasons={result['reasons']}")


Scenario 1: retrain=False  reasons=[]
Scenario 2: retrain=True  reasons=['volume: 600 new rows >= threshold 500']
Scenario 3: retrain=True  reasons=['performance: AUC dropped 0.063 (promotion=0.853, recent=0.79) >= threshold 0.03', 'drift: 4 consecutive drift flags >= threshold 3']
